# PROMISE Multi-Project NDG Extraction

This notebook runs and inspects the generalized Network Dependency Graph extraction pipeline implemented in `scripts/extract_promise_ndg.py`.

The script is the single source of truth. The notebook only wraps execution and validates the generated graph/tensor outputs.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SCRIPT_PATH = REPO_ROOT / 'scripts' / 'extract_promise_ndg.py'
INPUT_CSV = REPO_ROOT / 'outputs' / 'promise' / 'promise_preprocessed_standard.csv'
NDG_DIR = REPO_ROOT / 'outputs' / 'promise' / 'ndg'
SUMMARY_JSON = NDG_DIR / 'ndg_summary.json'
GRAPH_INDEX = NDG_DIR / 'graph_index.csv'
REPORT_MD = NDG_DIR / 'ndg_report.md'
VALIDATION_JSON = NDG_DIR / 'validation_issues.json'

print('Repository:', REPO_ROOT)
print('NDG extractor:', SCRIPT_PATH)
print('Input CSV:', INPUT_CSV)
print('NDG output directory:', NDG_DIR)

## 1. Design

The extractor creates one project-level NDG per PROMISE dataset.

Each NDG stores:

- nodes: mapped Java classes/files from the preprocessed dataset
- node features: the 20 preprocessed metric columns only
- node labels: binary defect labels in `y`
- typed dependency edges: `EXTENDS`, `IMPLEMENTS`, `FIELD_TYPE`, `PARAMETER_TYPE`, `RETURN_TYPE`, `OBJECT_CREATION`, `METHOD_CALL`

This graph format is designed for future node-level prediction: each file node has its own feature vector and target label.

In [ ]:
pre_df = pd.read_csv(INPUT_CSV)
print('combined preprocessing shape:', pre_df.shape)
print('datasets:', sorted(pre_df['dataset_name'].unique()))
print('mapped rows:', int(pre_df['source_path'].notna().sum()))
pre_df.head()

## 2. Run NDG Extraction

Terminal equivalent:

```bash
python scripts/extract_promise_ndg.py
```

To run one dataset only:

```bash
python scripts/extract_promise_ndg.py --dataset-name ant-1.6
```

In [ ]:
result = subprocess.run(
    [sys.executable, str(SCRIPT_PATH)],
    cwd=REPO_ROOT,
    text=True,
    capture_output=True,
    check=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)

## 3. Extraction Summary

In [ ]:
summary = json.loads(SUMMARY_JSON.read_text())
summary_view = {
    key: summary[key]
    for key in [
        'input_rows',
        'mapped_rows',
        'graphs_generated',
        'total_nodes',
        'total_edges',
        'node_feature_dim',
        'edge_type_vocab_size',
        'defective_nodes',
        'non_defective_nodes',
        'fallback_parses',
        'parse_failures',
        'validation_issues',
    ]
}
summary_view

## 4. Per-Project Results

In [ ]:
pd.DataFrame(summary['datasets'])

## 5. Graph Index

In [ ]:
graph_index = pd.read_csv(GRAPH_INDEX)
print('graph_index shape:', graph_index.shape)
graph_index

## 6. Inspect One NDG and Tensor Set

In [ ]:
example = graph_index.iloc[0]
print(example[['graph_id', 'dataset_name', 'num_nodes', 'num_edges', 'node_feature_dim']])

graph = json.loads(Path(example['graph_json']).read_text())
x = np.load(example['x_npy'])
y = np.load(example['y_npy'])
edge_index = np.load(example['edge_index_npy'])
edge_type = np.load(example['edge_type_npy'])

print('x shape:', x.shape)
print('y shape:', y.shape)
print('edge_index shape:', edge_index.shape)
print('edge_type shape:', edge_type.shape)
print('first nodes:')
pd.DataFrame(graph['nodes']).head()

In [ ]:
print('first edges:')
pd.DataFrame(graph['edges']).head(10)

## 7. Validation Checks

In [ ]:
validation_issues = json.loads(VALIDATION_JSON.read_text())
checks = {
    'ten_ndgs_generated': summary['graphs_generated'] == 10,
    'graph_index_rows_match_summary': len(graph_index) == summary['graphs_generated'],
    'total_nodes_match_mapped_rows': summary['total_nodes'] == summary['mapped_rows'],
    'node_feature_dim_is_20': summary['node_feature_dim'] == 20,
    'validation_issues_count_matches': len(validation_issues) == summary['validation_issues'],
}
checks

## 8. Report Preview

In [ ]:
report_text = REPORT_MD.read_text()
print('
'.join(report_text.splitlines()[:90]))